# PyTorch Fundamentals for NLP - Part III_WIP
> This blog post explains how to build a LSTM based Seq2Seq architecture for translating human-readable date to machine-readable date (e.g.: february 6 2019 --> 2019-02-06)
- toc: true
- branch: master
- author: Senthil Kumar
- badges: true
- comments: true
- categories: [pytorch_for_nlp]
- image: images/pytorch_nn/pytorch_offl_image.png
- hide: false

## 1. Introduction

In this blog piece, let us cover how we can build a
- `machine translation` application using an Encoder-Decoder architecture built using LSTM and linear units

## 2.Representing Text as Tensors - A Quick Recap

**Typical Process of Embedding Creation** <br>
- `text_data` >> `tokens` >> `numericals` >> `embeddings` 

## 3. A LSTM-based Machine Translation Pipeline

┣━━ **1.Loading dataset** <br>
┃   ┣━━ define `CustomTextDataset (torch.utils.data.Dataset)` <br>
┣━━ **2.Create Tokenizer** <br>
┣━━ **3.Build vocabulary** <br>
┣━━ **4.Create `Embeddings` related functions**<br>
┃   ┣━━ Create `collate_fn` to create mini-batch wise padded length source_batch and target_batch <br>
┣━━ **5.Create train, validation and test `DataLoaders`**<br>
┣━━ **6.Define `Model_Architectures`**<br>
┃   ┣━━ Define `EncoderRNN`, `DecoderRNN` and `Seq2SeqModel` <br>
┣━━ **7.define `training_loop` and `testing_loop` functions**<br>
┣━━ **8.Train the model and Evaluate on Test Data**<br>
┣━━ **9.Test the model on sample text**<br>

Importing basic modules

In [34]:
#collapse-hide
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data.dataset import random_split

import torchtext
# from torch.text.data.utils import get_tokenizer # not needed for this human to machine translation
from torchtext.vocab import build_vocab_from_iterator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### 3.1. Loading CSV data file as a PyTorch Dataset

In [2]:
import pandas as pd
date_dataset_pd = pd.read_csv('translation_pairs.csv',index_col=False)

In [3]:
date_dataset_pd.head()

,Human Readable Date,Machine Readable Date
0,february 6 2019,2019-02-06
1,26/07/1985,1985-07-26
2,mar 25 2010,2010-03-25
3,december 15 1976,1976-12-15
4,aug 8 1991,1991-08-08


In [19]:
class DateDataset(Dataset):
    def __init__(self, 
                 file_dir_w_file_name,
                 source_column,
                 target_column
                ):
        # supercharge your sub-class by inheriting the defaults from Parent class
        super(DateDataset,self).__init__()
        self.data_df = pd.read_csv(file_dir_w_file_name,index_col=False)
        self.source_text_list = list(self.data_df[source_column])
        self.target_text_list = list(self.data_df[target_column])
        
    def __len__(self):
        return len(self.data_df)
    
    def __getitem__(self,idx):
        return (self.source_text_list[idx], self.target_text_list[idx])

In [9]:
help(random_split)

Help on function random_split in module torch.utils.data.dataset:

random_split(dataset: torch.utils.data.dataset.Dataset[~T], lengths: Sequence[int], generator: Union[torch._C.Generator, NoneType] = <torch._C.Generator object at 0x7f4a98fdb610>) -> List[torch.utils.data.dataset.Subset[~T]]
    Randomly split a dataset into non-overlapping new datasets of given lengths.
    Optionally fix the generator for reproducible results, e.g.:
    
    >>> random_split(range(10), [3, 7], generator=torch.Generator().manual_seed(42))
    
    Args:
        dataset (Dataset): Dataset to be split
        lengths (sequence): lengths of splits to be produced
        generator (Generator): Generator used for the random permutation.



In [20]:
complete_date_dataset = DateDataset(file_dir_w_file_name='./translation_pairs.csv',
                 source_column='Human Readable Date',
                 target_column='Machine Readable Date')

In [21]:
len(complete_date_dataset)

30000

In [22]:
complete_date_dataset[67]

('27 march 2020', '2020-03-27')

**Notes**:
- There are 30K dates in this dataset. 
- The dataset was created using `faker` augmentation library using steps mentioned in [this Kaggle Notebook](https://www.kaggle.com/code/bkanupam/date-translation-with-encoder-decoder)

Let us do a 80-10-10 split

In [26]:
total_data_length = len(complete_date_dataset)
num_train_samples = int(len(complete_date_dataset) * 0.80)
num_valid_samples = int(len(complete_date_dataset) * 0.10)
num_test_samples = total_data_length - num_train_samples - num_valid_samples 
split_train_, split_valid_, split_test_ = random_split(complete_date_dataset,
                                                     [num_train_samples, 
                                                      num_valid_samples,
                                                      num_test_samples
                                                     ]
                                                    )

In [27]:
type(split_train_)

torch.utils.data.dataset.Subset

In [28]:
split_train_.indices[0:5]

[3730, 18844, 13554, 25302, 4160]

In [16]:
num_test_samples

3000

In [35]:
train_dataset = Subset(complete_date_dataset, split_train_)

In [42]:
train_dataset = []
sorted_indices = sorted(split_train_.indices)
for i, (x, y) in enumerate(complete_date_dataset):
    if i in sorted_indices:
        train_dataset.append((x,y))

In [43]:
train_dataset[0]

('february 6 2019', '2019-02-06')

In [44]:
len(train_dataset)

24000

## 3.2. Create Tokenizer

In [30]:
list('27 march 2020')

['2', '7', ' ', 'm', 'a', 'r', 'c', 'h', ' ', '2', '0', '2', '0']

In [31]:
# simply split the text into characters
tokenizer = lambda x: list(x)

In [57]:
for source_text, target_text  in train_dataset:
    print(source_text, target_text)
    print(tokenizer(source_text), tokenizer(target_text))
    break

february 6 2019 2019-02-06
['f', 'e', 'b', 'r', 'u', 'a', 'r', 'y', ' ', '6', ' ', '2', '0', '1', '9'] ['2', '0', '1', '9', '-', '0', '2', '-', '0', '6']


### 3.3 Building Vocabulary

In [91]:
#collapse-show
UNKNOWN_IDX = 0
PAD_IDX = 1
BOS_IDX = 2
EOS_IDX = 3

# Make sure the tokens are in order of their indices to properly insert them in vocab
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

def _yield_tokens(data_iter, dataset_type):
    for source_text, target_text  in data_iter:
        if dataset_type == 'source':
            yield tokenizer(source_text)
        elif dataset_type == 'target':
            yield tokenizer(target_text)

def create_vocab(train_dataset, dataset_type):
    print(f"Building {dataset_type} Vocabulary ...")
    vocab = build_vocab_from_iterator(_yield_tokens(train_dataset,dataset_type), 
                                      min_freq=1,
                                      specials=special_symbols,
                                      special_first=True
                                     )
    return vocab

In [92]:
src_vocab = create_vocab(train_dataset, 'source')
tgt_vocab = create_vocab(train_dataset, 'target')

Building source Vocabulary ...
Building target Vocabulary ...


In [93]:
len(src_vocab)

38

In [94]:
len(tgt_vocab)

15

In [95]:
tgt_vocab(tokenizer('2020-03-27'))

[7, 5, 7, 5, 4, 5, 11, 4, 7, 10]

In [67]:
tgt_vocab.get_stoi()

{'5': 14,
 '4': 13,
 '6': 12,
 '3': 11,
 '<eos>': 3,
 '<bos>': 2,
 '<unk>': 0,
 '-': 4,
 '0': 5,
 '8': 9,
 '7': 10,
 '1': 6,
 '<pad>': 1,
 '2': 7,
 '9': 8}

In [66]:
tgt_vocab.get_itos()

['<unk>',
 '<pad>',
 '<bos>',
 '<eos>',
 '-',
 '0',
 '1',
 '2',
 '9',
 '8',
 '7',
 '3',
 '6',
 '4',
 '5']

In [68]:
src_vocab.get_stoi()

{'w': 37,
 'v': 36,
 'h': 33,
 'f': 32,
 'l': 31,
 'p': 30,
 '4': 27,
 '6': 26,
 'c': 25,
 '/': 29,
 'o': 23,
 '0': 8,
 '2': 7,
 '<pad>': 1,
 'j': 24,
 '9': 6,
 't': 19,
 'e': 9,
 's': 21,
 'g': 35,
 'a': 10,
 'y': 13,
 '<bos>': 2,
 '<eos>': 3,
 '<unk>': 0,
 ' ': 4,
 'r': 11,
 'u': 12,
 '8': 14,
 'i': 34,
 '7': 15,
 'm': 16,
 '1': 5,
 'n': 17,
 'd': 18,
 'b': 20,
 '5': 28,
 '3': 22}

In [96]:
t = torch.tensor(src_vocab(tokenizer(train_dataset[0][0])),
                                     dtype=torch.int64
                                                   )
torch.nn.functional.pad(t,(0,20),mode='constant',value=1)

tensor([32,  9, 20, 11, 12, 10, 11, 13,  4, 26,  4,  7,  8,  5,  6,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1])

### 3.4 Create Padify function

In [100]:
def _add_bos_eos_tokens(tokenized_numericalized_text):
    return torch.cat([torch.tensor([BOS_IDX]),
                      tokenized_numericalized_text,
                      torch.tensor([EOS_IDX]),
                     ])

def padify(batch):
    # batch is a list of (source, target) pair of tuples
    # padding both source and target sequences
    source_list, target_list = [], []
    for source_text, target_text in batch:
        tokenized_numericalized_source_text = torch.tensor(src_vocab(tokenizer(source_text)),
                                                    dtype=torch.int64
                                                   )
        final_tn_source_text = _add_bos_eos_tokens(tokenized_numericalized_source_text)
        source_list.append(final_tn_source_text)
        
        tokenized_numericalized_target_text = torch.tensor(tgt_vocab(tokenizer(target_text)),
                                                    dtype=torch.int64
                                                   )
        final_tn_target_text = _add_bos_eos_tokens(tokenized_numericalized_target_text)
        target_list.append(final_tn_target_text)
    
    # padding source sequence
    max_source_length = max(map(len, source_list))
    source_list_overall = torch.stack([torch.nn.functional.pad(torch.tensor(t),
                                                               (0,max_source_length - len(t)),
                                                               mode='constant',
                                                               value=1) for t in source_list
                                                              ])
    # padding target sequence
    max_target_length = max(map(len, target_list))
    target_list_overall = torch.stack([torch.nn.functional.pad(torch.tensor(t),
                                                               (0,max_target_length - len(t)),
                                                               mode='constant',
                                                               value=1) for t in target_list
                                                              ])
    return source_list_overall.to(device), target_list_overall.to(device)

### 3.5 Prepare DataLoaders

In [101]:
BATCH_SIZE = 8

train_dataloader = DataLoader(split_train_,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              collate_fn=padify
                             ) 

valid_dataloader = DataLoader(split_valid_,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              collate_fn=padify
                             )

test_dataloader = DataLoader(split_test_,
                             batch_size=BATCH_SIZE,
                             shuffle=True,
                             collate_fn=padify
                            )

In [102]:
for i, (source_transformed, target_transformed) in enumerate(test_dataloader):
    print(f"Tracking batch {i}")
    print("Source Text Shape:",source_transformed.shape)
    print("Target Text Shape:",target_transformed.shape)
    if i==3:
        break
    print("****")

Tracking batch 0
Source Text Shape: torch.Size([8, 21])
Target Text Shape: torch.Size([8, 12])
****
Tracking batch 1
Source Text Shape: torch.Size([8, 26])
Target Text Shape: torch.Size([8, 12])
****
Tracking batch 2
Source Text Shape: torch.Size([8, 25])
Target Text Shape: torch.Size([8, 12])
****
Tracking batch 3
Source Text Shape: torch.Size([8, 19])
Target Text Shape: torch.Size([8, 12])


/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).


### 3.6 Define Model Architectures

- Let us a build a Seq2Seq Model with 3 classes - `EncoderRNN`, `DecoderRNN` and `Seq2SeqModel`
- Let us build a 2 layer LSTM Encoder and a 2 layer LSTM Decoder

#### 3.6.1 EncoderRNN

Encoder has the following `layers` and their corresponding `parameters`: <br>
**1.Encoder_Embedding_Layer**:<br> Converts the one-hot encoded source vector (in our case, called `tokenized_numericalized__source_text`) into dense `embeddings`
- `input_dim` - Equal to the Source Vocabulary Dimension (`len(src_vocab)`)
- `emb_dim` - dimensionality of the embedding_layer
**2.LSTM_Layers**:<br>
- `emb_dim` - (same explanation as above)
- `hid-dim` -  dimensionality of the hidden and cell states of the LSTM
- `n_layers`- number of layers in the LSTM unit
- `dropout` - A regularization parameter to prevent overfitting

#### 3.6.2 DecoderRNN

Decoder has the following `layers` and their corresponding `parameters`: <br>
**1.Decoder_Embedding_Layer**:<br> Converts the one-hot encoded source vector (in our case, called `tokenized_numericalized__target_text`) into dense `embeddings`
- `output_dim` - Equal to the Target Vocabulary Dimension (`len(tgt_vocab)`)
- `emb_dim` - dimensionality of the embedding_layer
**2.LSTM_Layers**:<br>
- `emb_dim` - (same explanation as above)
- `hid-dim` -  dimensionality of the hidden and cell states of the LSTM
- `n_layers`- number of layers in the LSTM unit
- `dropout` - A regularization parameter to prevent overfitting


### 3.6.3 Seq2SeqModel

Questions to Answer: 
- Why `Hidden dimensions of encoder and decoder must be equal`
- Why `Encoder and decoder must have equal number of layers!`
- What is `teacher_forcing_ratio`

## 3.7 Setting Hyper-parameters, define `training_loop` and `testing_loop` functions

## 3.8 Training the Seq2Seq Model

## 3.9 Testing on Sample Text

## 4. Conclusion

Sources <br>

- PyTorch Seq2Seq Model (codes written using old PyTorch <0.8) | [link](https://github.com/bentrevett/pytorch-seq2seq/blob/master/1%20-%20Sequence%20to%20Sequence%20Learning%20with%20Neural%20Networks.ipynb)
- Data Pipeline inspiration | [link](https://pytorch.org/tutorials/beginner/translation_transformer.html)
- An Encoder-Decoder Pipeline written differently than above pipeline in Kaggle | [link](https://www.kaggle.com/code/bkanupam/date-translation-with-encoder-decoder)

<hr>